In [1]:
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import h3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from shapely.geometry import box
from shapely.geometry import Point

In [2]:
df = pd.read_csv('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/final_datasets/london_final_dataset.csv')

In [3]:
categories = ['traffic_count',
       'bike_amenities', 'car_amenities', 'children_amenity', 'food_amenities',
       'health_amenity', 'home_amenities', 'infrastructure_amenities',
       'leisure_amenities', 'needs_amenities', 'other_amenities',
       'pet_amenity', 'public_transport', 'shops_amenities',
       'social_amenities', 'study_amenity', 'water_amenities', 'work_amenity',
     'highway']

In [4]:
output_dir = '/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps/amenity'

for category in categories:
    moscow_map = folium.Map(location=[51.485, -0.12], zoom_start=12)
    marker_cluster = MarkerCluster().add_to(moscow_map)
    for idx, row in df.iterrows():
        if row[category]:
            folium.Marker(
                    location=[row.latitude, row.longitude],
                    popup=row.get('amenity', f'{category}'),
                    tooltip=row[category],
                    icon=folium.Icon(color='blue', icon='info-sign')
                ).add_to(marker_cluster)
    output_path = os.path.join(output_dir, f"london_{category}_map.html")
    moscow_map.save(output_path)
    print(f"Карта {category} сохранена")

Карта traffic_count сохранена
Карта bike_amenities сохранена
Карта car_amenities сохранена
Карта children_amenity сохранена
Карта food_amenities сохранена
Карта health_amenity сохранена
Карта home_amenities сохранена
Карта infrastructure_amenities сохранена
Карта leisure_amenities сохранена
Карта needs_amenities сохранена
Карта other_amenities сохранена
Карта pet_amenity сохранена
Карта public_transport сохранена
Карта shops_amenities сохранена
Карта social_amenities сохранена
Карта study_amenity сохранена
Карта water_amenities сохранена
Карта work_amenity сохранена
Карта highway сохранена


In [5]:
parking_data = gpd.read_file('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/infrastructure/GB-CITY-001-221a8b52-20250213-ru-gpkg/data/parking-polygon.gpkg')
parking_data.columns = parking_data.columns.str.lower()

In [6]:
parking_map = folium.Map(location=[51.485, -0.12], zoom_start=12, tiles='cartodb positron', max_bounds=True)

for _, row in parking_data.iterrows():
    folium.GeoJson(row['geometry'], style_function=lambda x: {'color': 'blue', 'opacity': 0.5}).add_to(parking_map)

output_dir = '/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps'
output_path = os.path.join(output_dir, f"london_parkings.html")
parking_map.save(output_path)

In [7]:
# трафик
AVERAGE_ANNUAL_DAILY_FLOW_PATH = "/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/traffic_data/london_data/dft_aadf_region_id_6.csv"
daily_flow = pd.read_csv(AVERAGE_ANNUAL_DAILY_FLOW_PATH).dropna(subset=['latitude', 'longitude'])

In [8]:
grouped_data = daily_flow.groupby(['latitude', 'longitude', 'year'], as_index=False).agg({'all_motor_vehicles': 'sum'})

In [9]:
output_dir = '/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps/traffic'

for year in grouped_data['year'].unique():
    yearly_data = grouped_data[grouped_data['year'] == year]
    center_lat, center_lon = yearly_data[['latitude', 'longitude']].mean()
    london_map = folium.Map(location=[center_lat, center_lon], zoom_start=10)
    marker_cluster = MarkerCluster().add_to(london_map)
    for _, row in yearly_data.iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Трафик: {row['all_motor_vehicles']}",
            tooltip=f"Трафик: {row['all_motor_vehicles']}",
            icon=folium.Icon(color='blue')
        ).add_to(marker_cluster)
    
    output_path = os.path.join(output_dir, f"london_traffic_{year}_year_map.html")
    london_map.save(output_path)
    print(f"Карта за {year} сохранена")

Карта за 2019 сохранена
Карта за 2018 сохранена
Карта за 2020 сохранена
Карта за 2021 сохранена
Карта за 2022 сохранена
Карта за 2023 сохранена
Карта за 2000 сохранена
Карта за 2001 сохранена
Карта за 2002 сохранена
Карта за 2003 сохранена
Карта за 2004 сохранена
Карта за 2005 сохранена
Карта за 2006 сохранена
Карта за 2007 сохранена
Карта за 2008 сохранена
Карта за 2009 сохранена
Карта за 2010 сохранена
Карта за 2011 сохранена
Карта за 2012 сохранена
Карта за 2013 сохранена
Карта за 2014 сохранена
Карта за 2015 сохранена
Карта за 2016 сохранена
Карта за 2017 сохранена


In [10]:
import folium
from folium.plugins import MarkerCluster
import os

average_traffic = daily_flow.groupby(['latitude', 'longitude'], as_index=False).agg({'all_motor_vehicles': 'mean'})
center_lat, center_lon = average_traffic[['latitude', 'longitude']].mean()
traffic_map = folium.Map(location=[center_lat, center_lon], zoom_start=10)
marker_cluster = MarkerCluster().add_to(traffic_map)
for _, row in average_traffic.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Средний трафик: {int(row['all_motor_vehicles'])}",
        tooltip=f"Трафик: {int(row['all_motor_vehicles'])}",
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

output_path = os.path.join(output_dir, f"london_traffic_mean_map.html")
london_map.save(output_path)

In [11]:
power_line = gpd.read_file('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/infrastructure/GB-CITY-001-221a8b52-20250213-ru-gpkg/data/power-line.gpkg')
power_line.columns = power_line.columns.str.lower()

In [12]:
output_dir = '/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps'
nsw_map = folium.Map(location=[51.485, -0.12], zoom_start=12)
folium.GeoJson(power_line, name="Power Lines").add_to(nsw_map)
output_path = os.path.join(output_dir, f"london_power_lines_map.html")
nsw_map.save(output_path)

In [13]:
chargers_map = folium.Map(location=[51.485, -0.12], zoom_start=12)
marker_cluster = MarkerCluster().add_to(chargers_map)

for _, row in df.iterrows():
    if row['has_charging_station']:
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Stations: {row['number_stations']}",
            icon=folium.Icon(color='blue', icon='bolt', prefix='fa'),
            tooltip="Charging Station"
        ).add_to(chargers_map)

output_path = os.path.join(output_dir, f"london_chargers_map.html")
chargers_map.save(output_path)

In [14]:
combined_map = folium.Map(location=[51.485, -0.12], zoom_start=12)

# 1. Добавляем зарядные станции (кластеризованные маркеры)
charger_cluster = MarkerCluster(name="Charging Stations").add_to(combined_map)

for _, row in df.iterrows():
    if row['has_charging_station']:
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Stations: {row['number_stations']}",
            icon=folium.Icon(color='blue', icon='bolt', prefix='fa'),
            tooltip="Charging Station"
        ).add_to(charger_cluster)

power_line = gpd.read_file('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/infrastructure/GB-CITY-001-221a8b52-20250213-ru-gpkg/data/power-line.gpkg')
power_line.columns = power_line.columns.str.lower()

def style_function(feature):
    return {
        'color': 'red',
        'weight': 3,
        'opacity': 0.7
    }

# Добавляем линии на карту
folium.GeoJson(
    power_line,
    name="Power Lines",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['voltage'], aliases=['Voltage:'])
).add_to(combined_map)

# 3. Добавляем управление слоями
folium.LayerControl().add_to(combined_map)

# Сохраняем карту
output_path = os.path.join(output_dir, "london_combined_chargers_power_lines.html")
combined_map.save(output_path)

print(f"Комбинированная карта сохранена: {output_path}")

Комбинированная карта сохранена: /Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps/london_combined_chargers_power_lines.html


In [6]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
output_dir = '/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps'

df = pd.read_csv('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/final_datasets/london_final_dataset.csv')

In [7]:
idx_0 = np.where(df['has_charging_station'] == 0)[0]
idx_1 = np.where(df['has_charging_station'] == 1)[0]

n_samples = min(int(0.3 * len(idx_0)), int(0.3 * len(idx_1)), int(0.5 * 0.3 * len(df['has_charging_station'])))

rng = np.random.RandomState(RANDOM_STATE)
test_idx_0 = rng.choice(idx_0, size=n_samples, replace=False)
test_idx_1 = rng.choice(idx_1, size=n_samples, replace=False)

test_idx = np.concatenate([test_idx_0, test_idx_1])
train_idx = np.setdiff1d(np.arange(len(df['has_charging_station'])), test_idx)

df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

In [8]:
combined_map = folium.Map(location=[51.485, -0.12], zoom_start=12)

# 1. Добавляем зарядные станции (кластеризованные маркеры)
charger_cluster = MarkerCluster(name="Charging Stations").add_to(combined_map)

for _, row in df_train.iterrows():
    if row['has_charging_station']:
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Stations: {row['number_stations']}",
            icon=folium.Icon(color='blue', icon='bolt', prefix='fa'),
            tooltip="Charging Station"
        ).add_to(charger_cluster)

power_line = gpd.read_file('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/infrastructure/GB-CITY-001-221a8b52-20250213-ru-gpkg/data/power-line.gpkg')
power_line.columns = power_line.columns.str.lower()

def style_function(feature):
    return {
        'color': 'red',
        'weight': 3,
        'opacity': 0.7
    }

# Добавляем линии на карту
folium.GeoJson(
    power_line,
    name="Power Lines",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['voltage'], aliases=['Voltage:'])
).add_to(combined_map)

# 3. Добавляем управление слоями
folium.LayerControl().add_to(combined_map)

# Сохраняем карту
output_path = os.path.join(output_dir, "london_combined_chargers_power_lines_train.html")
combined_map.save(output_path)

print(f"Комбинированная карта сохранена: {output_path}")

Комбинированная карта сохранена: /Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps/london_combined_chargers_power_lines_train.html


In [9]:
combined_map = folium.Map(location=[51.485, -0.12], zoom_start=12)

# 1. Добавляем зарядные станции (кластеризованные маркеры)
charger_cluster = MarkerCluster(name="Charging Stations").add_to(combined_map)

for _, row in df_test.iterrows():
    if row['has_charging_station']:
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Stations: {row['number_stations']}",
            icon=folium.Icon(color='blue', icon='bolt', prefix='fa'),
            tooltip="Charging Station"
        ).add_to(charger_cluster)

power_line = gpd.read_file('/Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/data/infrastructure/GB-CITY-001-221a8b52-20250213-ru-gpkg/data/power-line.gpkg')
power_line.columns = power_line.columns.str.lower()

def style_function(feature):
    return {
        'color': 'red',
        'weight': 3,
        'opacity': 0.7
    }

# Добавляем линии на карту
folium.GeoJson(
    power_line,
    name="Power Lines",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['voltage'], aliases=['Voltage:'])
).add_to(combined_map)

# 3. Добавляем управление слоями
folium.LayerControl().add_to(combined_map)

# Сохраняем карту
output_path = os.path.join(output_dir, "london_combined_chargers_power_lines_test.html")
combined_map.save(output_path)

print(f"Комбинированная карта сохранена: {output_path}")

Комбинированная карта сохранена: /Users/dmitrystarukhin/final_diploma_mipt_sumgf/london/map_per_feature/maps/london_combined_chargers_power_lines_test.html
